In [37]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [38]:
df = pd.read_csv("maha_oc_s2_merged.csv")
print(df.shape)

(26033, 212)


/var/folders/lm/z783vpwd6zjfsgvb0mdpm4b80000gn/T/ipykernel_13301/1027882284.py:1: DtypeWarning: Columns (71) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("maha_oc_s2_merged.csv")


In [39]:
df.head()

,sample_date_clean,fid,TH_LAT,TH_LONG,CLIMATE_VALUE,CLIMATE_SUBCLASS,CLIMATE_CLASS,DOMSOI,SOIL_TYPE,SOIL_SUBCLASS,...,ts_BSI_w4,ts_NDVI_w5,ts_BSI_w5,ts_NDVI_w6,ts_BSI_w6,ts_NDVI_w7,ts_BSI_w7,ts_NDVI_w8,ts_BSI_w8,base_n
0,17-02-2025,48980.0,15.709677,74.047631,2.0,Am,Tropical,Ap,ACRISOLS,Plinthic Acrisols,...,0.248097,0.477888,0.057228,0.747465,-0.231050,0.404198,0.084964,0.307499,0.211454,72
1,17-02-2025,48981.0,15.709731,74.048897,2.0,Am,Tropical,Ap,ACRISOLS,Plinthic Acrisols,...,0.243333,0.417055,0.142600,0.641459,-0.146411,0.308770,0.160615,0.258200,0.241939,69
2,17-02-2025,48982.0,15.711065,74.047657,2.0,Am,Tropical,Ap,ACRISOLS,Plinthic Acrisols,...,0.327024,0.321039,0.244666,0.750917,-0.210699,0.159889,0.282719,0.193916,0.295636,73
3,31-03-2024,48839.5,15.804995,73.733783,2.0,Am,Tropical,Nd,NITOSOLS,Dystric Nitosols,...,0.288295,0.185185,0.289969,0.303691,0.237000,0.767790,-0.264913,0.151415,0.269820,74
4,31-03-2024,48835.0,15.805012,73.734112,2.0,Am,Tropical,Nd,NITOSOLS,Dystric Nitosols,...,0.261415,0.205882,0.253479,0.299266,0.203720,0.618922,-0.149223,0.173198,0.276983,74


In [40]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [41]:
import numpy as np
import pandas as pd

sequence_cols = [
    "ts_NDVI_w1", "ts_BSI_w1",
    "ts_NDVI_w2", "ts_BSI_w2",
    "ts_NDVI_w3", "ts_BSI_w3",
    "ts_NDVI_w4", "ts_BSI_w4",
    "ts_NDVI_w5", "ts_BSI_w5",
    "ts_NDVI_w6", "ts_BSI_w6",
    "ts_NDVI_w7", "ts_BSI_w7",
    "ts_NDVI_w8", "ts_BSI_w8",
]

target = "OC"

data = df[sequence_cols + [target]].copy()

# Remove rows with missing values
data = data.dropna()

print(data.shape)

(23736, 17)


In [42]:
X = data[sequence_cols].values
y = data[target].values

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (23736, 16)
y shape: (23736,)


In [43]:
X = X.reshape(-1, 8, 2)

print(X.shape)

(23736, 8, 2)


In [44]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [45]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_flat = X_train.reshape(-1, 2)
X_test_flat = X_test.reshape(-1, 2)

X_train_flat = scaler.fit_transform(X_train_flat)
X_test_flat = scaler.transform(X_test_flat)

X_train = X_train_flat.reshape(X_train.shape)
X_test = X_test_flat.reshape(X_test.shape)

In [46]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    LSTM(64, input_shape=(8, 2)),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dense(1)
])

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model.summary()

/Users/aditibharadwaj/Documents/experiqs/MH_SOC/dl_env/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        17,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,265 (75.25 KB)

 Trainable params: 19,265 (75.25 KB)

 Non-trainable params: 0 (0.00 B)

In [47]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
475/475 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.1864 - mae: 0.2815 - val_loss: 0.1686 - val_mae: 0.2676
Epoch 2/100
475/475 ━━━━━━━━━━━━━━━━━━━━ 0s 991us/step - loss: 0.1703 - mae: 0.2678 - val_loss: 0.1670 - val_mae: 0.2785
Epoch 3/100
475/475 ━━━━━━━━━━━━━━━━━━━━ 0s 993us/step - loss: 0.1673 - mae: 0.2656 - val_loss: 0.1641 - val_mae: 0.2598
Epoch 4/100
475/475 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1667 - mae: 0.2649 - val_loss: 0.1649 - val_mae: 0.2730
Epoch 5/100
475/475 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1657 - mae: 0.2639 - val_loss: 0.1634 - val_mae: 0.2611
Epoch 6/100
475/475 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1644 - mae: 0.2634 - val_loss: 0.1638 - val_mae: 0.2528
Epoch 7/100
475/475 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.1635 - mae: 0.2624 - val_loss: 0.1635 - val_mae: 0.2648
Epoch 8/100
475/475 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1632 - mae: 0.2631 - val_loss: 0.1623 - val_mae: 0.2678
Epoch 9/100
475/475 ━━━━━━━━━━━━━━━━━━━━ 0s 

In [48]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = model.predict(X_test).flatten()

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R²   : {r2:.4f}")

149/149 ━━━━━━━━━━━━━━━━━━━━ 0s 605us/step
RMSE : 0.3772
MAE  : 0.2392
R²   : 0.2040
